# Supplier efficiency (BCC DEA) and interactive trade-off targets (DEA-IMRP)
12 suppliers · inputs: unit price, late delivery %, shipping error % · outputs: product-quality and customer-service utilities (from the Evidential Reasoning assessment).

No external data is needed: the data is defined below.

## 1. Data and input-oriented BCC model
For each supplier o: min θ s.t. Σλⱼxᵢⱼ ≤ θ·xᵢₒ, Σλⱼyᵣⱼ ≥ yᵣₒ, Σλⱼ = 1, λ ≥ 0.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

suppliers = list("ABCDEFGHIJKL")
# Inputs (less is better)
X = np.array([
    # unit price ($), late delivery (%), shipping error (%)
    [1.34, 3, 8], [1.44, 5, 4], [2.13, 6, 6], [2.01, 9, 3],
    [3.44, 4, 7], [2.98, 4, 6], [1.34, 1, 3], [5.10, 2, 3],
    [1.40, 1, 3], [1.50, 6, 4], [1.33, 3, 3], [1.94, 9, 6],
])
# Outputs (more is better): utilities from the Evidential Reasoning assessment
Y = np.array([
    # product quality, customer service
    [0.7524, 0.6525], [0.6941, 0.6491], [0.5544, 0.6198], [0.5433, 0.5318],
    [0.4625, 0.5402], [0.5010, 0.3958], [0.1363, 0.4677], [0.7998, 0.6926],
    [0.4523, 0.6108], [0.4360, 0.6128], [0.3778, 0.5172], [0.5372, 0.5513],
])


def bcc_input_oriented(X, Y, o):
    """Return (theta, lambdas) for DMU o. Decision vector = [theta, lambda_1..lambda_n]."""
    n, m = X.shape
    s = Y.shape[1]
    c = np.r_[1, np.zeros(n)]                        # minimise theta
    A_in = np.c_[-X[o].reshape(m, 1), X.T]           # sum lambda*x - theta*x_o <= 0
    A_out = np.c_[np.zeros((s, 1)), -Y.T]            # -sum lambda*y <= -y_o
    A_ub = np.r_[A_in, A_out]
    b_ub = np.r_[np.zeros(m), -Y[o]]
    A_eq = np.r_[0, np.ones(n)].reshape(1, -1)       # convexity (BCC)
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=[1],
                  bounds=[(0, None)] * (n + 1), method="highs")
    return res.x[0], res.x[1:]

## 2. Efficiency scores

In [2]:
rows = []
for o, name in enumerate(suppliers):
    theta, lam = bcc_input_oriented(X, Y, o)
    peers = {suppliers[j]: round(l, 3) for j, l in enumerate(lam) if l > 1e-4 and j != o}
    targets = lam @ X                                 # composite benchmark inputs
    rows.append({
        "Supplier": name,
        "theta*": round(theta, 4),
        "Status": "Efficient" if theta > 0.9999 else "Inefficient",
        "Peers (lambda)": peers if theta < 0.9999 else "-",
        "Target unit price": round(targets[0], 3),
        "Target late del %": round(targets[1], 3),
        "Target ship err %": round(targets[2], 3),
    })

results = pd.DataFrame(rows)
pd.set_option("display.width", 200, "display.max_colwidth", 60)
print(results.to_string(index=False))
print(f"\nEfficient suppliers: {', '.join(results.loc[results.Status=='Efficient','Supplier'])}")

Supplier  theta*      Status                                   Peers (lambda)  Target unit price  Target late del %  Target ship err %
       A  1.0000   Efficient                                                -              1.340              3.000              8.000
       B  1.0000   Efficient                                                -              1.440              5.000              4.000
       C  0.6564 Inefficient {'A': 0.131, 'B': 0.284, 'I': 0.507, 'K': 0.078}              1.398              2.554              3.938
       D  1.0000   Efficient                                                -              2.010              9.000              3.000
       E  0.4287 Inefficient {'B': 0.001, 'D': 0.088, 'H': 0.006, 'I': 0.905}              1.475              1.715              3.001
       F  0.5223 Inefficient  {'B': 0.134, 'D': 0.066, 'H': 0.03, 'I': 0.771}              1.556              2.089              3.134
       G  1.0000   Efficient                           

## 3. DEA-IMRP: minimax reference point with gradient projection
Frontier Ω = {λ ≥ 0 : Σλy ≥ yₒ, Σλx ≤ xₒ, Σλ = 1}.

1. Payoff table → ideal point f\*
2. Minimax LP with w = 1/(x − f\*) → iter-0 efficient point
3. Normal N = w·β (LP duals); project the preference df onto the tangent plane: Δ = df − (df·N / N·N)·N
4. Step α = γ·α_max, new target Y = f + αΔ, w = 1/(Y − f\*), re-solve
5. Stop when ‖α·Δ‖ falls below the noise threshold

In [3]:
INPUTS = ["Unit Price", "Late Delivery", "Ship Error"]

# Decision-maker preferences: priority Late Delivery > Ship Error > Unit Price, risk-averse
DF_SCHEDULE = [np.array([+0.05, -0.50, -0.20]),   # iter-1
               np.array([+0.04, -0.30, -0.30]),   # iter-2
               np.array([+0.03, -0.25, -0.18])]   # iter-3
GAMMA = [0.25, 0.20, 0.15]
FINAL_ITER = {"C": 3, "E": 1, "F": 2, "J": 1, "L": 2}   # stopping iteration from the analysis


def _omega(jo):
    """Constraints defining Omega_jo, for decision vector lambda (length n)."""
    n = X.shape[0]
    A_ub = np.r_[-Y.T, X.T]          # outputs: -sum lambda*y <= -y_jo ; inputs: sum lambda*x <= x_jo
    b_ub = np.r_[-Y[jo], X[jo]]
    A_eq = np.ones((1, n))
    return A_ub, b_ub, A_eq, np.array([1.0])


def payoff_ideal(jo):
    """Ideal point f*: minimum of each input objective taken separately."""
    A_ub, b_ub, A_eq, b_eq = _omega(jo)
    return np.array([
        linprog(X[:, i], A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                bounds=(0, None), method="highs").fun
        for i in range(X.shape[1])])


def minimax(jo, w, f_star):
    """min eta  s.t. w_i (f_i(lambda) - f*_i) <= eta,  lambda in Omega_jo.
    Returns (f(lambda), lambda, beta) where beta are the duals of the minimax constraints."""
    n, m = X.shape
    A_ub0, b_ub0, A_eq0, b_eq = _omega(jo)
    c = np.r_[1.0, np.zeros(n)]                                  # vars: [eta, lambda]
    A_mm = np.c_[-np.ones((m, 1)), (w[:, None] * X.T)]           # w_i*sum lambda x_i - eta <= w_i f*_i
    A_ub = np.r_[A_mm, np.c_[np.zeros((A_ub0.shape[0], 1)), A_ub0]]
    b_ub = np.r_[w * f_star, b_ub0]
    A_eq = np.c_[np.zeros((1, 1)), A_eq0]
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq,
                  bounds=[(None, None)] + [(0, None)] * n, method="highs")
    lam = res.x[1:]
    beta = -res.ineqlin.marginals[:m]
    return lam @ X, lam, beta


def run_imrp(jo, n_iter):
    x0 = X[jo]
    f_star = payoff_ideal(jo)
    w = 1.0 / (x0 - f_star)
    f, lam, beta = minimax(jo, w, f_star)
    log = [{"iter": 0, "target": f.copy(), "lambda": lam, "move": np.nan}]
    for t in range(n_iter):
        N = w * beta                                             # normal vector (Yang & Li 2002, Eq. 21)
        df = DF_SCHEDULE[t]
        delta = df - (df @ N) / (N @ N) * N                      # projection (Eq. 28)
        caps = [(f[i] - f_star[i]) / -delta[i] if delta[i] < 0 else
                (x0[i] - f[i]) / delta[i] for i in range(len(f)) if abs(delta[i]) > 1e-12]
        alpha = GAMMA[t] * min(caps)
        move = np.linalg.norm(alpha * delta)
        Y_new = f + alpha * delta
        w = 1.0 / (Y_new - f_star)                               # weight update (Yang et al. 2009, Eq. 29)
        f, lam, beta = minimax(jo, w, f_star)
        log.append({"iter": t + 1, "target": f.copy(), "lambda": lam, "move": move})
    return f_star, log

## 4. Run for the five inefficient suppliers

In [4]:
pd.set_option("display.width", 200)
summary = []
for jo, name in enumerate(suppliers):
    theta, _ = bcc_input_oriented(X, Y, jo)
    if theta > 0.9999:
        continue
    f_star, log = run_imrp(jo, FINAL_ITER[name])
    print(f"\n=== Supplier {name}  (BCC theta* = {theta:.4f}) ===")
    print("Ideal point f*:", np.round(f_star, 4))
    traj = pd.DataFrame([{"iter": r["iter"], **dict(zip(INPUTS, np.round(r["target"], 4))),
                          "||alpha*Delta||": round(r["move"], 4)} for r in log])
    print(pd.concat([pd.DataFrame([{"iter": "current", **dict(zip(INPUTS, X[jo]))}]), traj]).to_string(index=False))
    final = log[-1]["target"]
    summary.append({"Supplier": name, **{f"{k} (current)": v for k, v in zip(INPUTS, X[jo])},
                    **{f"{k} (target)": round(v, 2) for k, v in zip(INPUTS, final)},
                    "Final iter": FINAL_ITER[name]})
print("\n=== Final improvement targets ===")
print(pd.DataFrame(summary).to_string(index=False))


=== Supplier C  (BCC theta* = 0.6564) ===
Ideal point f*: [1.352  1.4185 3.1229]
   iter  Unit Price  Late Delivery  Ship Error  ||alpha*Delta||
current      2.1300         6.0000      6.0000              NaN
      0      1.5053         2.3210      3.6897              NaN
      1      1.6615         2.1085      3.6386           0.2686
      2      1.7552         2.0287      3.5511           0.1511
      3      1.8114         1.9696      3.5118           0.0905

=== Supplier E  (BCC theta* = 0.4287) ===
Ideal point f*: [1.3323 1.0294 3.    ]
   iter  Unit Price  Late Delivery  Ship Error  ||alpha*Delta||
current      3.4400         4.0000      7.0000              NaN
      0      1.3989         1.1233      3.1265              NaN
      1      1.4020         1.1023      3.1211           0.0253

=== Supplier F  (BCC theta* = 0.5223) ===
Ideal point f*: [1.3333 1.1401 3.    ]
   iter  Unit Price  Late Delivery  Ship Error  ||alpha*Delta||
current      2.9800         4.0000      6.0000    